# Acceptance: calibrating several models jointly

**This is a sign-off page, not a tutorial.** It exercises roadmap step 50 (composability
package `CP9`, findings `CX46`, `CX47`, `CX48`): the `BayesianModel` helpers `check_params`,
`log_likelihood(strict=True)`, `sample_params`, `factor`, `local_params` and
`posterior_runs(params_fn=)`. Two city epidemics and one natural-history cohort each have their
own `BayesianModel`, and all three have a parameter called `recovery`. The joint model ties the
three recovery rates together with a hierarchy on `log recovery_m`, centred for the data-rich
cohort and non-centred for the cities. Nothing here asserts; the automated checks live in
`tests/test_calib_multi_model.py::test_gate_*`.

Run every cell top to bottom (about three minutes). Each section ends with a **What to check** box.

| | You are checking that… |
|---|---|
| 1 | all three models have a `recovery`, while the joint sites are prefixed and there is no bare `recovery` |
| 2 | a misspelled name silently falls back to `fixed_params` unless `strict=True` |
| 3 | the joint fit pins city A's recovery far tighter than city A alone, with the truth inside |
| 4 | each model replays from the joint posterior in one line, and the bands cover the data |

Pitfalls this page does not hide:

- `wf.lhs` stratifies only parentless scalar sites; plate offsets and hierarchical children are
  filled by `Predictive`.
- Within one compiled model a name has one value: two values of "the same" parameter need two
  names or a stratified parameter, not prefixes.
- Two `BayesianModel`s sampled with the same prefix collide in numpyro's duplicate-site error, so
  the failure is loud.
- Choose centring per member. A member whose own data barely inform it (the cities) is written
  non-centred (`z ~ Normal(0, 1)`, value `exp(mu + tau * z)`); a member whose data pin it (the
  cohort) is written centred (`log value ~ Normal(mu, tau)`). The wrong choice funnels at small
  `tau`: with the cohort non-centred, warmup here fails its divergence check. See
  `futureplans/hierarchical-centring-diagnostics.md`.
- A given value replaces the model's prior, support included. Use a solver that stays sound at
  any rate the hierarchy can reach (these models use the default adaptive solver).

In [ ]:
import warnings

import jax.numpy as jnp
import numpy as np
import numpyro
import numpyro.distributions as dist
import optax
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from numpyro.infer import MCMC, NUTS

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from dataclasses import dataclass
from typing import Any

from summer4 import (
    Compartments,
    CompiledModel,
    FlowModel,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Target,
    TargetSet,
    TransitionFlow,
)
from summer4.epi import FOIKind, ForceOfInfection, MixingMatrix
from summer4.epi.calibration import (
    BayesianModel,
    NegativeBinomial,
    NumpyroPosterior,
    Poisson,
    Uniform,
)
from summer4.epi.calibration import workflow as wf

## Models and data

**City models A and B** share one compiled frequency-dependent SIR with `Param("infection")` and
`Param("recovery")`. Their populations are 1e6 and 4e5, with 20 initial infectious each, and they
are fit to infectious counts every two days over days 0–20 (`NegativeBinomial`). Each has
`fixed_params={"recovery": 0.25}`, a development default that makes the silent-fallback pitfall
visible. **The cohort model** is a different compiled model (`I → R` only, 300 infectious at the
start, `Poisson` counts on days 0–14) with no fixed parameters. The joint model samples
`mu ~ Normal(log(0.25), 1)` and `tau ~ HalfNormal(0.3)`. Each city gets a non-centred
`z_m ~ Normal(0, 1)` with `log recovery_m = mu + tau * z_m`; the cohort, whose data pin its rate,
gets a centred `log_recovery_cohort ~ Normal(mu, tau)`. It hands `recovery = exp(log recovery_m)` to each model through `sample_params(f"{m}.", given=...)`, and
scores it with `factor`. The cell below is a verbatim copy of `tests/gate_multi_model.py`.

In [ ]:
# --- shared settings (kept in sync with tests/gate_multi_model.py) ---
# --- constants (notebook cells must match) ---
DATA_SEED = 11
TRUE_MU = float(np.log(0.3))
TRUE_TAU = 0.25
TRUE_INFECTION = {"a": 0.55, "b": 0.75}
MU_PRIOR_LOC = float(np.log(0.25))
MU_PRIOR_SCALE = 1.0
TAU_PRIOR_SCALE = 0.3
NB_DISPERSION = 20.0
POP_SIZES = {"a": 1.0e6, "b": 4.0e5}
COHORT_START = 300.0
FIXED_RECOVERY = 0.25
MODELS = ("a", "b", "cohort")
OPT_SEED = 0
LHS_SEED = 0
MCMC_SEED = 3
MCMC_CHAINS = 2
MCMC_SAMPLES_PER_CHAIN = 200
MCMC_WARMUP_START = 60
MCMC_WARMUP_MIN = 80
# Two or three members leave tau weakly determined; at 0.9 the divergence rate sits near the
# warmup check's 1% limit and flips between platforms. 0.99 takes smaller steps.
MCMC_TARGET_ACCEPT = 0.99
OPT_MAX_STEPS = 300
OPT_CHUNK = 50
OPT_LR = 0.05
LHS_M = 48
N_OPT_STARTS = 8

CITY_TIMES = np.arange(0.0, 21.0, 2.0)
COHORT_TIMES = np.arange(0.0, 15.0, 1.0)


def build_city_model() -> tuple[CompiledModel, Property, PropertyMap]:
    """Frequency-dependent SIR with ``Param("infection")`` and ``Param("recovery")``."""
    state = Property("state", ("S", "I", "R"))
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    flow_model = FlowModel(pmap)
    flow_model.add_flow(
        TransitionFlow(
            "infection",
            state["S"],
            state["I"],
            ForceOfInfection(
                "infection",
                infectious=state["I"],
                group_by=pop,
                mixing=MixingMatrix(pop, np.array([[1.0]])),
                kind=FOIKind.FREQUENCY,
                contact_rate=Param("infection"),
            ),
        )
    )
    flow_model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
    return flow_model.compile(), state, pmap


def build_cohort_model() -> tuple[CompiledModel, Property, PropertyMap]:
    """A natural-history cohort: states ``I`` and ``R`` only, one flow ``I -> R``."""
    state = Property("state", ("I", "R"))
    pmap = PropertyMap.from_property(state)
    flow_model = FlowModel(pmap)
    flow_model.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery")))
    return flow_model.compile(), state, pmap


@dataclass(frozen=True)
class GateBundle:
    models: dict[str, BayesianModel]
    truth: dict[str, float]
    observed: dict[str, np.ndarray]
    times: dict[str, np.ndarray]
    joint_model: Any


def build_gate() -> GateBundle:
    """Two city models (shared compiled graph) and a cohort model, plus the joint model."""
    rng = np.random.default_rng(DATA_SEED)
    z_true = dict(zip(MODELS, rng.normal(size=len(MODELS)), strict=True))
    true_recovery = {m: float(np.exp(TRUE_MU + TRUE_TAU * z_true[m])) for m in MODELS}

    city_cm, city_state, city_pmap = build_city_model()
    run_kwargs = {"t0": 0.0, "t1": 20.0, "dt": 0.25}  # default adaptive solver
    models: dict[str, BayesianModel] = {}
    observed: dict[str, np.ndarray] = {}
    for m in ("a", "b"):
        y0_values = np.zeros(city_pmap.size)
        y0_values[city_pmap.select(city_state["S"])] = POP_SIZES[m] - 20.0
        y0_values[city_pmap.select(city_state["I"])] = 20.0
        y0 = PropertyData.wrap(city_pmap, y0_values)
        infectious = Compartments(where=city_state["I"])
        clean = city_cm.run(
            {"infection": TRUE_INFECTION[m], "recovery": true_recovery[m]},
            y0,
            save=SavePlan(requests={"I": SaveRequest(infectious, ts=CITY_TIMES)}),
            **run_kwargs,
        )
        expected = np.asarray(clean["I"].values.data).reshape(-1)
        observed[m] = rng.negative_binomial(
            NB_DISPERSION, NB_DISPERSION / (NB_DISPERSION + expected)
        ).astype(float)
        models[m] = BayesianModel(
            city_cm,
            {"recovery": FIXED_RECOVERY},
            priors=(Uniform("infection", 0.1, 1.5), Uniform("recovery", 0.02, 1.0)),
            targets=TargetSet(
                targets=(
                    Target(
                        key="I",
                        times=CITY_TIMES,
                        values=observed[m],
                        quantity=infectious,
                        likelihood=NegativeBinomial(dispersion=NB_DISPERSION),
                    ),
                )
            ),
            y0=y0,
            run_kwargs=run_kwargs,
        )

    cohort_cm, cohort_state, cohort_pmap = build_cohort_model()
    c_y0_values = np.zeros(cohort_pmap.size)
    c_y0_values[cohort_pmap.select(cohort_state["I"])] = COHORT_START
    c_y0 = PropertyData.wrap(cohort_pmap, c_y0_values)
    c_infectious = Compartments(where=cohort_state["I"])
    c_kwargs = {"t0": 0.0, "t1": 14.0, "dt": 0.25}
    c_clean = cohort_cm.run(
        {"recovery": true_recovery["cohort"]},
        c_y0,
        save=SavePlan(requests={"I": SaveRequest(c_infectious, ts=COHORT_TIMES)}),
        **c_kwargs,
    )
    c_expected = np.asarray(c_clean["I"].values.data).reshape(-1)
    observed["cohort"] = rng.poisson(c_expected).astype(float)
    models["cohort"] = BayesianModel(
        cohort_cm,
        {},
        priors=(Uniform("recovery", 0.02, 1.0),),
        targets=TargetSet(
            targets=(
                Target(
                    key="I",
                    times=COHORT_TIMES,
                    values=observed["cohort"],
                    quantity=c_infectious,
                    likelihood=Poisson(),
                ),
            )
        ),
        y0=c_y0,
        run_kwargs=c_kwargs,
    )

    def joint_model() -> None:
        mu = numpyro.sample("mu", dist.Normal(MU_PRIOR_LOC, MU_PRIOR_SCALE))
        tau = numpyro.sample("tau", dist.HalfNormal(TAU_PRIOR_SCALE))
        # The cities' short series barely inform their own recovery, so they are non-centred.
        # The cohort's 300 people pin its recovery, so it is centred: non-centred, it funnels
        # at small tau and warmup fails its divergence check.
        log_recovery = {
            "a": mu + tau * numpyro.sample("z_a", dist.Normal(0.0, 1.0)),
            "b": mu + tau * numpyro.sample("z_b", dist.Normal(0.0, 1.0)),
            "cohort": numpyro.sample("log_recovery_cohort", dist.Normal(mu, tau)),
        }
        for m in MODELS:
            given = {"recovery": jnp.exp(log_recovery[m])}
            params = models[m].sample_params(f"{m}.", given=given)
            models[m].factor(m, params)

    truth = {f"{m}.recovery": true_recovery[m] for m in MODELS}
    truth.update({f"{m}.infection": v for m, v in TRUE_INFECTION.items()})
    truth.update({"mu": TRUE_MU, "tau": TRUE_TAU})
    times = {"a": CITY_TIMES, "b": CITY_TIMES, "cohort": COHORT_TIMES}
    return GateBundle(models, truth, observed, times, joint_model)


gate = build_gate()
models = gate.models
truth = gate.truth
joint = NumpyroPosterior(gate.joint_model)

pd.Series({k: round(v, 4) for k, v in truth.items()}, name="generating values")

## 1. Local names and site names

Each model's parameters are local names; the joint model's sites carry a prefix.

In [ ]:
pd.DataFrame(
    {
        "model": list(models) + ["joint"],
        "prior_names()": [models[m].prior_names() for m in models] + [joint.prior_names()],
    }
).set_index("model")

> **What to check**
>
> - `a`, `b` and `cohort` each list a `recovery` (the cohort lists only that).
> - The joint row lists `mu`, `tau`, `z_a`, `z_b`, `log_recovery_cohort`, `a.infection` and `b.infection`, and no bare
>   `recovery`: the three recovery rates enter through `given=` and are recorded as
>   deterministic sites `a.recovery`, `b.recovery` and `cohort.recovery`.

## 2. Pitfall: a misspelled name used to fall back silently

`log_likelihood` merges the dict you pass over `fixed_params`. Spell `recovery` wrong and the model
quietly scores the fixed 0.25. With `strict=True` (what `factor` always uses), `check_params` names
both the missing and the unknown parameter.

In [ ]:
bm_a = models["a"]
truth_rec_a = truth["a.recovery"]
infection_a = truth["a.infection"]
lenient_typo = float(bm_a.log_likelihood({"infection": infection_a, "recovry": truth_rec_a}))
correct = float(bm_a.log_likelihood({"infection": infection_a, "recovery": truth_rec_a}))
try:
    bm_a.log_likelihood({"infection": infection_a, "recovry": truth_rec_a}, strict=True)
    strict_message = "(no error)"
except ValueError as err:
    strict_message = str(err)

pd.DataFrame(
    {
        "log-likelihood / message": [lenient_typo, correct, strict_message],
    },
    index=[
        'lenient, "recovry" (used the fixed 0.25)',
        'correct, "recovery" = truth',
        'strict, "recovry"',
    ],
)

> **What to check**
>
> - The lenient misspelled call returns a finite number that differs from the correct one.
> - The strict message names `recovry` (unknown) and `recovery` (missing).

## 3. Joint fit against city A alone

The same pipeline on both: `wf.lhs` → `wf.evaluate` → `wf.optimize` → `wf.warmup_until` →
`wf.sample_until`. City A alone sees only a short growth phase, so `infection` and `recovery` lie
along a ridge. The joint model lets cohort and city B inform the shared hierarchy.

NUTS starts from the optimiser's end points, not the LHS design. The models use summer4's
default adaptive solver: a fixed-step Euler solve goes unstable at large rates, which puts
cliffs in the likelihood that a hierarchical value (an unbounded `exp(mu + tau * z)`) can reach.


In [ ]:
warm_rule = wf.WarmupRule(min_warmup=MCMC_WARMUP_MIN, rhat=1.15)
stop = wf.StopRule(rhat=None, ess=None, max_samples=MCMC_SAMPLES_PER_CHAIN)


def fit(post):
    design = wf.evaluate(post, wf.lhs(post, LHS_M, seed=LHS_SEED), batch_size=16)
    starts = design.best(N_OPT_STARTS)
    opt = wf.optimize(
        post,
        starts,
        method=wf.Optax(optax.adam(OPT_LR)),
        chunk_steps=OPT_CHUNK,
        max_steps=OPT_MAX_STEPS,
        seed=OPT_SEED,
    )

    def make_mcmc(num_warmup: int) -> MCMC:
        return MCMC(
            NUTS(post.numpyro_model(), target_accept_prob=MCMC_TARGET_ACCEPT),
            num_warmup=num_warmup,
            num_samples=MCMC_SAMPLES_PER_CHAIN,
            num_chains=MCMC_CHAINS,
        )

    warm = wf.warmup_until(
        make_mcmc,
        MCMC_WARMUP_START,
        warm_rule,
        init_params=opt.candidates.init_params(MCMC_CHAINS, seed=MCMC_SEED),
        seed=MCMC_SEED,
    )
    print(f"{type(post).__name__}: warmup {warm.num_warmup} steps, converged={warm.converged}")
    return wf.sample_until(warm.mcmc, stop, seed=MCMC_SEED)


run_joint = fit(joint)
run_a = fit(bm_a)

alone = np.asarray(run_a.samples["recovery"]).reshape(-1)
joint_rec = {m: np.asarray(run_joint.samples[f"{m}.recovery"]).reshape(-1) for m in MODELS}

fig = go.Figure()
fig.add_box(y=alone, name="A alone", marker_color="gray")
for m in MODELS:
    fig.add_box(y=joint_rec[m], name=f"{m} (joint)")
fig.add_scatter(
    x=["A alone"] + [f"{m} (joint)" for m in MODELS],
    y=[truth["a.recovery"]] + [truth[f"{m}.recovery"] for m in MODELS],
    mode="markers",
    marker={"symbol": "star", "size": 14, "color": "gold", "line": {"width": 1}},
    name="truth",
)
fig.update_layout(
    title="3. Posterior recovery rate: city A alone vs the joint fit",
    xaxis_title="model",
    yaxis_title="recovery rate",
    height=520,
)
fig.show()

rows = {"A alone": (alone, truth["a.recovery"])}
rows.update({f"{m} (joint)": (joint_rec[m], truth[f"{m}.recovery"]) for m in MODELS})
pd.DataFrame(
    {
        name: {
            "10%": float(np.quantile(x, 0.1)),
            "50%": float(np.quantile(x, 0.5)),
            "90%": float(np.quantile(x, 0.9)),
            "width": float(np.quantile(x, 0.9) - np.quantile(x, 0.1)),
            "truth": t,
            "truth in 10–90%": bool(np.quantile(x, 0.1) <= t <= np.quantile(x, 0.9)),
        }
        for name, (x, t) in rows.items()
    }
).T

> **What to check**
>
> - Both fits print `converged=True`: the warmup passed its divergence and R-hat check.
> - The gray box (city A alone) is wide; the joint box for A is much narrower
>   (width under half).
> - Each gold star lies inside its box's 10–90% interval (`truth in 10–90%` is `True`).

## 4. Replay each model from the joint posterior

`posterior_runs(idata, params_fn=...)` flattens the joint draws (plate sites allowed), maps them to
each model's local names with `local_params`, checks them strictly and runs the forward model. No
arrays are built by hand.

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"model {m}" for m in MODELS])
for col, m in enumerate(MODELS, start=1):
    bm = models[m]
    runs = bm.posterior_runs(
        run_joint.idata,
        n=100,
        params_fn=lambda d, bm=bm, m=m: bm.local_params(d, f"{m}."),
    )
    band = runs.quantiles((0.1, 0.5, 0.9))["baseline"]["I"]
    t = band.index.to_numpy()
    fig.add_scatter(x=t, y=band["0.9"], mode="lines", line={"width": 0}, showlegend=False, row=1, col=col)
    fig.add_scatter(
        x=t, y=band["0.1"], mode="lines", line={"width": 0}, fill="tonexty",
        fillcolor="rgba(99,110,250,0.25)", name="10–90%", showlegend=(col == 1), row=1, col=col,
    )
    fig.add_scatter(x=t, y=band["0.5"], mode="lines", line={"color": "rgb(99,110,250)"},
                    name="median", showlegend=(col == 1), row=1, col=col)
    fig.add_scatter(x=gate.times[m], y=gate.observed[m], mode="markers",
                    marker={"color": "black", "size": 6}, name="observed", showlegend=(col == 1),
                    row=1, col=col)
    fig.update_xaxes(title_text="day", row=1, col=col)
    fig.update_yaxes(title_text="infectious", row=1, col=col)
fig.update_layout(title="4. Joint-posterior replay through each member model", height=460)
fig.show()

> **What to check**
>
> - The 10–90% band covers the observed points in all three panels.
> - The replay cell contains no hand-built arrays: one `params_fn` line per model.